# The minimum password length as a decision, not a constant

This notebook **talks to the system running in Docker** (nothing is stubbed), walks the film's
sequence scene by scene, **asserts every step** and **screenshots** the gallery.

It needs the stack up:

```bash
cd ~/Documents/git/portal && ./infra-up.sh
```

Then `Run All`. The last cell prints a PASS/FAIL table and lists the screenshots in `shots/`.
The notebook cleans up after itself: the `security_settings` row goes, and so does the compose
override it writes for the property scene.

In [ ]:
# --- configuration and helpers -------------------------------------------------
import json, re, subprocess, time, pathlib, datetime
import requests
from IPython.display import HTML, Image, display

SEC      = "http://localhost:8080"
MEMES    = "http://localhost:8083"
MAILPIT  = "http://localhost:8025"
PROJECT  = "security"                      # one compose project for the whole estate
PORTAL   = pathlib.Path.home() / "Documents/git/portal"
HERE     = pathlib.Path.cwd()
SHOTS    = HERE / "shots"; SHOTS.mkdir(exist_ok=True)
OVERRIDE = HERE / "compose.override.yml"   # written by the property scene, removed on the way out

ADMIN    = "admin@example.com"
PASSWORD = "StrongPassword1!"              # satisfies every rule of the policy
KEY      = "security.password.policy.min.length"
RUN      = datetime.datetime.now().strftime("%H%M%S")   # fresh e-mail addresses per run

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), str(detail)[:300])); 
    display(HTML(f"<div style='font-family:monospace;padding:2px 0'>"
                 f"<b style='color:{'#137333' if ok else '#c5221f'}'>{'PASS' if ok else 'FAIL'}</b> "
                 f"{name} <span style='color:#666'>{detail if not ok else ''}</span></div>"))
    return ok

def sh(*args, cwd=None, timeout=180):
    """Run a command and return its output; a non-zero exit is data, not an exception."""
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return (p.stdout + p.stderr).strip()

def psql(sql):
    """SQL straight against the security database - the way someone at the console writes it."""
    return sh("docker", "compose", "-p", PROJECT, "exec", "-T", "postgres",
              "psql", "-U", "postgres", "-d", "security", "-c", sql)

def show(title, response):
    """Render an HTTP response legibly enough to be read on camera."""
    try:    body = json.dumps(response.json(), indent=2, ensure_ascii=False)
    except Exception: body = response.text or "(empty body)"
    color = "#137333" if response.status_code < 400 else "#c5221f"
    display(HTML(f"<div style='font-family:monospace;font-size:13px;border-left:3px solid {color};"
                 f"padding:4px 10px;margin:4px 0;background:#fafafa'>"
                 f"<b>{title}</b> &rarr; <b style='color:{color}'>{response.status_code}</b>"
                 f"<pre style='margin:4px 0'>{body}</pre></div>"))
    return response

print("ready — SEC:", SEC, "| screenshot directory:", SHOTS)

## 0. Preflight — is the system up at all

In [ ]:
def health():
    rows = []
    for name, url in [("security (readiness)", f"{SEC}/health/readiness"),
                      ("memes / galeria",      f"{MEMES}/"),
                      ("Mailpit",              f"{MAILPIT}/")]:
        try:    code_ = requests.get(url, timeout=5).status_code
        except Exception as e: code_ = f"brak ({type(e).__name__})"
        rows.append((name, url, code_))
    db = psql("select 1;")
    rows.append(("postgres (security)", "docker compose exec postgres", "OK" if "1 row" in db else db[:80]))
    html = "".join(f"<tr><td style='padding:2px 12px 2px 0'>{n}</td>"
                   f"<td style='padding:2px 12px 2px 0;color:#666'>{u}</td>"
                   f"<td><b>{c}</b></td></tr>" for n, u, c in rows)
    display(HTML(f"<table style='font-family:monospace;font-size:13px'>{html}</table>"))
    return rows

rows = health()
check("the stack answers", all(str(c) in ("200", "OK") for _, _, c in rows),
      "bring it up: cd ~/Documents/git/portal && ./infra-up.sh")

## 1. A clean start — drop the row earlier runs left behind

Ladder levels: **live (database) > restart (property) > rebuild (`MinLength.DEFAULT` = 5)**. A vacant
live level is how the film begins — at the value the programmer shipped. This cell also removes a
compose override left by an interrupted run, so every run starts in the same place.

In [ ]:
# A previous run may have died between the property scene and the cleanup, leaving an override
# behind - the reset makes every run start from the same place no matter how the last one ended.
if OVERRIDE.exists():
    OVERRIDE.unlink()
    print(sh("docker", "compose", "-p", PROJECT, "-f", "docker-compose.yml", "up", "-d", "--no-build",
             "security", cwd=PORTAL, timeout=300))

print(psql(f"DELETE FROM security_settings WHERE name = '{KEY}';"))
print(psql("SELECT * FROM security_settings;"))
check("the live level is vacant", "(0 rows)" in psql("SELECT * FROM security_settings;"))

r = requests.get(f"{SEC}/health/readiness", timeout=5)
check("security answers after the reset", r.status_code == 200, r.text)

## 2. The ADMIN account — register, take the link from Mailpit, verify, sign in

In [ ]:
def mail_token(to, tries=20):
    """Pull the token out of the latest mail to this address (Mailpit is the dev inbox)."""
    for _ in range(tries):
        msgs = requests.get(f"{MAILPIT}/api/v1/search", params={"query": f"to:{to}"}, timeout=5).json()["messages"]
        if msgs:
            text = requests.get(f"{MAILPIT}/api/v1/message/{msgs[0]['ID']}", timeout=5).json()["Text"]
            m = re.search(r"(?:token|verify)=([A-Za-z0-9_\-]+)", text)
            if m: return m.group(1)
        time.sleep(1)
    return None

def register(email, password): return requests.post(f"{SEC}/register", json={"email": email, "password": password}, timeout=15)
def authenticate(email, password): return requests.post(f"{SEC}/authenticate", json={"email": email, "password": password}, timeout=15)

def ensure_admin():
    """The ADMIN is designated in compose (SECURITY_BOOTSTRAP_ADMINS); here we only create the account."""
    r = authenticate(ADMIN, PASSWORD)
    if r.status_code == 200:
        return r.json()["accessToken"]
    register(ADMIN, PASSWORD)                       # 201 for a taken address too (anti-enumeration)
    token = mail_token(ADMIN)
    if token:
        requests.post(f"{SEC}/verify-email", json={"token": token}, timeout=15)
    r = authenticate(ADMIN, PASSWORD)
    return r.json()["accessToken"] if r.status_code == 200 else None

TOKEN = ensure_admin()
AUTH  = {"Authorization": f"Bearer {TOKEN}"}
check("the ADMIN is signed in", TOKEN is not None, "check SECURITY_BOOTSTRAP_ADMINS in compose")

## 3. The ladder report and the ADMIN's hand

`GET /admin/settings/password/min-length` answers not only **what** but **which level answered** and
**what was refused on the way**. `POST` takes a step-up: the policy binds every future password, so a
live session is not proof enough.

In [ ]:
def report():
    return show("GET /admin/settings/password/min-length",
                requests.get(f"{SEC}/admin/settings/password/min-length", headers=AUTH, timeout=15)).json()

def step_up():
    return requests.post(f"{SEC}/account/step-up", headers=AUTH,
                         json={"action": "admin-settings", "password": PASSWORD}, timeout=15)

def set_min_length(value):
    """An elevation is one-shot and per action - buy a fresh one right before each write."""
    assert step_up().status_code == 200, "the step-up did not go through"
    return show(f"POST min-length = {value}",
                requests.post(f"{SEC}/admin/settings/password/min-length", headers=AUTH,
                              json={"value": value}, timeout=15))

r = report()
check("5 is in force, from the default level", r["value"] == 5 and "rebuild" in r["source"], r)

### Scene 1 — the programmer ships a default

A five-character password (with a digit, a capital and a special character) is accepted, because 5
is the length in force.

In [ ]:
r = show("POST /register (five-character password)", register(f"u1-{RUN}@example.com", "Ab1!x"))
check("a five-character password is accepted", r.status_code == 201, r.text)

### Scene 2 — a deployment claims the "restart" level

The `security.password.policy.min.length` property (here as an environment variable) covers the
default — but only after a restart. The notebook writes a compose override, restarts the service
and waits for readiness.

In [ ]:
def restart_security(min_length=None):
    """None = no property (the starting state); a number = the deployment claims the restart level."""
    if min_length is None:
        OVERRIDE.write_text("services:\n  security:\n    environment: {}\n")
    else:
        OVERRIDE.write_text("services:\n  security:\n    environment:\n"
                            f"      SECURITY_PASSWORD_POLICY_MIN_LENGTH: \"{min_length}\"\n")
    out = sh("docker", "compose", "-p", PROJECT, "-f", "docker-compose.yml", "-f", str(OVERRIDE),
             "up", "-d", "--no-build", "security", cwd=PORTAL, timeout=300)
    for _ in range(60):
        try:
            if requests.get(f"{SEC}/health/readiness", timeout=3).status_code == 200: return out
        except Exception: pass
        time.sleep(3)
    return out + " (readiness never came back)"

restart_security(8)
r = report()
check("the restart level answers, value 8", r["value"] == 8 and "restart" in r["source"], r)

resp = show("POST /register (seven-character password)", register(f"u2-{RUN}@example.com", "Ab1!xyz"))
check("seven characters refused, the message carries 8", resp.status_code == 422
      and any(e.get("MIN_LENGTH_NOT_MET") == 8 for e in resp.json()["passwordErrors"]), resp.text)

### Scene 3 — an ADMIN decides while the system runs

The live level covers the property. From that moment every place a password is established measures
against the new floor — and a refusal **names the minimum in force**, because the policy is live
configuration and a bare error code would leave the caller guessing.

In [ ]:
set_min_length(10)
r = report()
check("10 is in force, from the live level", r["value"] == 10 and "live" in r["source"], r)

resp = show("POST /register (nine-character password)", register(f"u3-{RUN}@example.com", "Nine1!aaa"))
check("nine characters refused, the message carries 10", resp.status_code == 422
      and any(e.get("MIN_LENGTH_NOT_MET") == 10 for e in resp.json()["passwordErrors"]), resp.text)

### Scene 4a — a length below the policy's own floor

The `MinLength` value object is the only gate. The refusal goes through the use case, so **nothing
changes**.

In [ ]:
resp = set_min_length(3)
check("3 is refused, with a reason", resp.status_code == 400 and resp.json()["status"] == "REFUSED", resp.text)

r = report()
check("after the refusal 10 is still in force", r["value"] == 10 and "live" in r["source"], r)

### Scene 4b — a row written straight into the database is not law

Writing at the psql console bypasses the gate — a breach of the contract, not a hole in the design
(an application database, in Fowler's sense). The ladder refuses the illegal level, **falls through**
and says so both in the report and in the log.

In [ ]:
print(psql(f"INSERT INTO security_settings (name, value) VALUES ('{KEY}', '3') "
           f"ON CONFLICT (name) DO UPDATE SET value = EXCLUDED.value, updated_at = now();"))
print(psql("SELECT name, value FROM security_settings;"))

r = report()
rejected = r.get("rejected", [])
check("the ladder fell to the property (8), the live level being illegal",
      r["value"] == 8 and "restart" in r["source"], r)
check("the report names the refused level and the reason",
      any(x["source"].startswith("live") and str(x["value"]) == "3" and "at least" in x["reason"] for x in rejected), rejected)

log = sh("docker", "compose", "-p", PROJECT, "logs", "--since", "3m", "security")
warn = [l for l in log.splitlines() if "min.length" in l.lower() and "warn" in l.lower()]
display(HTML("<pre style='font-size:12px;background:#fafafa;padding:6px'>" + ("\n".join(warn[-3:]) or "(no WARN line in the last 3 minutes)") + "</pre>"))
check("the service warns about the refused row in its log", bool(warn), "no WARN line")

### Scene 4c — and with no property the ladder falls all the way to the default

Drop the restart level: the illegal row stays in the database, and `MinLength.DEFAULT` = 5 is what the
system lives by.

In [ ]:
restart_security(None)
r = report()
check("5 is in force from the default level, the live level still refused",
      r["value"] == 5 and "rebuild" in r["source"] and any(x["source"].startswith("live") for x in r.get("rejected", [])), r)

resp = show("POST /register (five-character password)", register(f"u4-{RUN}@example.com", "Ab1!x"))
check("five characters are accepted again", resp.status_code == 201, resp.text)

### Scene 5 — the same thing, seen from the gallery

The browser shows the refusal the way a user meets it: **grouped per field**, carrying the policy
parameter in force for this very attempt. Screenshots land in `shots/`.

In [ ]:
# Jupyter already runs an asyncio loop, so Playwright uses the async API (top-level await).
from playwright.async_api import async_playwright

async def shot(page, name, width=760):
    path = SHOTS / f"{name}.png"
    await page.screenshot(path=str(path))
    display(Image(str(path), width=width))
    return path

pw      = await async_playwright().start()
browser = await pw.chromium.launch()
page    = await browser.new_page(viewport={"width": 1100, "height": 800}, device_scale_factor=2)

await page.goto(MEMES, wait_until="networkidle")
await shot(page, "01-gallery")

await page.get_by_role("tab", name="Create account").click()
# the browser no longer votes on addresses (noValidate), so the server's verdict is the only one
await page.get_by_label("e-mail").fill(f"ui-{RUN}@wp")
await page.get_by_label("password").fill("abc")                    # breaks several rules at once
await page.get_by_role("button", name="Create account").click()
await page.wait_for_selector("text=That will not do", timeout=15000)

alert = page.locator(".MuiAlert-root").first
text  = await alert.inner_text()
await shot(page, "02-registration-refused")
await alert.screenshot(path=str(SHOTS / "03-notice.png"))
display(Image(str(SHOTS / "03-notice.png"), width=520))

await browser.close(); await pw.stop()

print(text)
check("the notice has an e-mail section and a password section", "e-mail" in text and "password" in text, text)
check("the e-mail section carries the server's code", "domain missing dot" in text.lower(), text)
check("the password code carries the policy parameter", "min length not met: 5" in text, text)

## 6. Cleanup and summary

In [ ]:
print(psql(f"DELETE FROM security_settings WHERE name = '{KEY}';"))
OVERRIDE.unlink(missing_ok=True)
r = report()
check("after cleanup the default 5 is in force, nothing refused", r["value"] == 5 and not r.get("rejected"), r)

ok = sum(1 for _, passed, _ in CHECKS if passed)
rows = "".join(f"<tr><td style='color:{'#137333' if p else '#c5221f'};padding:2px 10px 2px 0'><b>{'PASS' if p else 'FAIL'}</b></td>"
               f"<td style='padding:2px 10px 2px 0'>{n}</td><td style='color:#666'>{d if not p else ''}</td></tr>"
               for n, p, d in CHECKS)
display(HTML(f"<h3>{ok}/{len(CHECKS)} checks passed</h3>"
             f"<table style='font-family:monospace;font-size:13px'>{rows}</table>"
             f"<p>Screenshots: {', '.join(sorted(p.name for p in SHOTS.glob('*.png')))}</p>"))